# Mechanics of LLM Memory: Activities Notebook

**Language:** Python  
**Topics:** RAIL, session isolation, Instance Amnesia, Replay Tax, RunnableWithMessageHistory, LangGraph  
**Level:** warm-up (read) to build (write)

### How this works
- Same offline setup as the demo. `USE_REAL_MODEL = False`, no API key needed.
- Each activity has: a brief, decision steps, a **starter** cell you edit, a **solution** cell, and a **check** cell that asserts you got it right.
- Progression is deliberate:

```mermaid
flowchart LR
    W[Warm up: read and spot] --> D[Debug: find and fix one line]
    D --> B[Build: write small pieces]
    B --> C[Capstone: an isolated trimmed copilot]
```

Run the setup, then go top to bottom. Do not peek at a solution until you have tried the starter.

## Setup

Run both cells. Reuse the same `LocalAgent`, `show`, and `model` as the demo.

In [1]:
# Run once. Safe to re-run. Restart the kernel if a fresh install changes versions.
%pip install -q langchain langchain-core langgraph
# Optional, only needed when USE_REAL_MODEL = True below:
# %pip install -q langchain-anthropic        # direct Anthropic API
# %pip install -q langchain-aws              # Claude via Amazon Bedrock
print("Dependencies ready.")

Note: you may need to restart the kernel to use updated packages.
Dependencies ready.


In [2]:
# === Toggle ===================================================================
# False -> offline LocalAgent. No API key. You SEE the notepad on every call.
# True  -> real Claude answers. Needs ANTHROPIC_API_KEY in the environment.
USE_REAL_MODEL = False
# =============================================================================

import re, warnings
from typing import List

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration

# Current LangChain marks RunnableWithMessageHistory as deprecated (it still runs).
# We use it because it makes the memory steps visible. The production path is
# LangGraph persistence, covered near the end. We silence only that one warning.
warnings.filterwarnings("ignore", message=".*RunnableWithMessageHistory.*")

class LocalAgent(BaseChatModel):
    # A tiny, transparent stand-in for a real LLM. No API key.
    # Its ONLY skills: remember a PNR it was told, spot a cancellation, and echo.
    # Purpose: make memory mechanics visible offline. It is NOT a smart model.
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        seen_text = " ".join(m.content for m in messages if isinstance(m.content, str))
        pnr_match = re.search(r"PNR\s+([A-Z0-9]{5,8})", seen_text)
        pnr = pnr_match.group(1) if pnr_match else None
        last_human = ""
        for m in reversed(messages):
            if m.type == "human" and isinstance(m.content, str):
                last_human = m.content
                break
        low = last_human.lower()
        if "pnr" in low and "?" in last_human:
            reply = f"Your PNR is {pnr}." if pnr else "I do not have your PNR. Could you share it?"
        elif "cancel" in low or "leg" in low:
            reply = "I see the BLR to DEL leg on your booking. I can help with that."
        elif last_human:
            reply = f"Noted: {last_human}"
        else:
            reply = "How can I help with your booking today?"
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=reply))])

    @property
    def _llm_type(self):
        return "local-agent"

def get_model():
    if USE_REAL_MODEL:
        # Direct Anthropic API. For Bedrock swap in:
        #   from langchain_aws import ChatBedrockConverse
        #   return ChatBedrockConverse(
        #       model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
        #       region_name="us-east-1")
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model="claude-haiku-4-5", temperature=0)
    return LocalAgent()

def show(title, messages):
    # Prints the exact list of messages the model will see. This IS the notepad.
    print(f"--- {title}: {len(messages)} message(s) the model will see ---")
    for i, m in enumerate(messages, 1):
        text = m.content if isinstance(m.content, str) else str(m.content)
        print(f"  {i}. {m.type:6} | {text}")

model = get_model()
print("Model in use:", type(model).__name__, "| USE_REAL_MODEL =", USE_REAL_MODEL)

Model in use: LocalAgent | USE_REAL_MODEL = False


Shared pipeline for the activities. Run once.

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise airline support agent."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | model | StrOutputParser()
print("Shared chain ready.")

Shared chain ready.


## Warm-up A1: trace the flow (no code)
**Read only.** Follow the diagram for a two turn chat, correct RAIL.

```mermaid
flowchart LR
    Q[user question] --> AP[append human]
    AP --> INV[invoke on full list]
    INV --> AL[append ai reply]
    AL --> NEXT[next turn]
```

**Question:** on turn 2, which messages does `invoke` receive? Pick one.
- (a) only turn 2's question
- (b) system, turn 1 human, turn 1 ai, turn 2 human
- (c) system and turn 2 human only

Write your letter in the next cell, then reveal the answer.

In [4]:
my_answer_A1 = "?"   # set to "a", "b", or "c"
print("A1:", my_answer_A1)

A1: ?


<details><summary>Reveal A1</summary>

**(b).** Correct RAIL appends the human turn, invokes on the full list, then logs the reply. By turn 2 the notepad holds system, both turn 1 messages, and the turn 2 question.
</details>

## Warm-up A2: read the code (no writing)
**Read only.** Given this line:

```python
chain = prompt | model | StrOutputParser()
```

**Question:** what does the pipe do? Pick one.
- (a) runs the three objects in parallel
- (b) feeds each stage's output into the next, left to right
- (c) picks whichever stage responds first

In [5]:
my_answer_A2 = "?"   # "a", "b", or "c"
print("A2:", my_answer_A2)

A2: ?


<details><summary>Reveal A2</summary>

**(b).** The pipe composes runnables. Output of `prompt` feeds `model`, whose output feeds `StrOutputParser`. Same interface at every stage is what makes this work.
</details>

## Warm-up A3: spot the missing step (identify, do not fix yet)
**Read only.** This loop has amnesia for its own replies.

```python
messages = [SystemMessage("Airline support.")]
def ask(user_text):
    messages.append(HumanMessage(user_text))
    reply = model.invoke(messages)
    return reply.content
```

**Question:** which RAIL step is missing? Set the variable to `"Retrieve"`, `"Augment"`, `"Invoke"`, or `"Log"`.

In [6]:
missing_step_A3 = "?"
print("A3:", missing_step_A3)

A3: ?


<details><summary>Reveal A3</summary>

**"Log".** The AI reply is never appended, so the model never sees its own answers on later turns.
</details>

## Debug A4: fix the broken loop (write one line)
**Brief:** the Head of CX reports the bot re-asks for details it already gave.

**Decision steps**
1. Which RAIL step is missing? (you found it in A3)
2. Add exactly one line so the AI reply is logged.

Edit the starter, then run the check.

In [ ]:
messages_A4 = [SystemMessage("You are a concise airline support agent.")]

def ask_A4(user_text):
    messages_A4.append(HumanMessage(user_text))
    reply = model.invoke(messages_A4)
    # TODO: add ONE line here so the reply survives to the next turn.
    return reply.content

ask_A4("I am Rao, PNR JX48Q2, Gold tier.")
ask_A4("What is my PNR?")

'Your PNR is JX48Q2.'

In [9]:
# SOLUTION A4
messages_A4 = [SystemMessage("You are a concise airline support agent.")]

def ask_A4(user_text):
    messages_A4.append(HumanMessage(user_text))
    reply = model.invoke(messages_A4)
    messages_A4.append(reply)        # the missing Log step
    return reply.content

ask_A4("I am Rao, PNR JX48Q2, Gold tier.")
print(ask_A4("What is my PNR?"))

Your PNR is JX48Q2.


In [10]:
# CHECK A4
ai_turns = [m for m in messages_A4 if m.type == "ai"]
assert len(ai_turns) == 2, "The AI replies are not being logged."
assert any("JX48Q2" in m.content for m in messages_A4), "The PNR is not in the notepad."
print("A4 passed: replies are logged and the PNR is remembered.")

A4 passed: replies are logged and the PNR is remembered.


## Debug A5: fix the ordering (identify and correct)
**Brief:** every answer is one turn late.

**Decision steps**
1. Find the line that invokes the model too early.
2. Reorder so the new question is appended before invoking.

In [9]:
messages_A5 = [SystemMessage("You are a concise airline support agent.")]

def ask_A5(user_text):
    reply = model.invoke(messages_A5)          # BUG: invoked before the new question is added
    messages_A5.append(HumanMessage(user_text))
    messages_A5.append(reply)
    return reply.content

# TODO: reorder the three lines above so RAIL runs Augment then Invoke then Log.

In [12]:
# SOLUTION A5
messages_A5 = [SystemMessage("You are a concise airline support agent.")]

def ask_A5(user_text):
    messages_A5.append(HumanMessage(user_text))   # Augment first
    reply = model.invoke(messages_A5)             # then Invoke
    messages_A5.append(reply)                     # then Log
    return reply.content

print(ask_A5("I am Rao, PNR JX48Q2."))
print(ask_A5("What is my PNR?"))

Noted: I am Rao, PNR JX48Q2.
Your PNR is JX48Q2.


In [13]:
# CHECK A5
# After two turns the last human question must sit before the final ai reply.
types = [m.type for m in messages_A5]
assert types[-2:] == ["human", "ai"], "Ordering is still wrong."
print("A5 passed: Augment, Invoke, Log in the right order.")

A5 passed: Augment, Invoke, Log in the right order.


## Debug A6: fix the key mismatch (find both bugs)
**Brief:** history is fetched but the bot still forgets.

**Decision steps**
1. The placeholder name and `history_messages_key` must match.
2. The `input_messages_key` must match the `{input}` slot.
Fix both in the starter.

In [12]:
bad_prompt = ChatPromptTemplate.from_messages([
    ("system", "Airline support."),
    MessagesPlaceholder("chat_history"),   # note the name
    ("human", "{question}"),               # note the variable
])
bad_chain = bad_prompt | model | StrOutputParser()

store_A6 = {}
def get_hist_A6(sid):
    if sid not in store_A6:
        store_A6[sid] = InMemoryChatMessageHistory()
    return store_A6[sid]

# TODO: fix the two keys so they match the prompt above.
bot_A6 = RunnableWithMessageHistory(
    bad_chain, get_hist_A6,
    input_messages_key="input",          # wrong on purpose
    history_messages_key="history",      # wrong on purpose
)

In [14]:
# SOLUTION A6 (self-contained)
bad_prompt = ChatPromptTemplate.from_messages([
    ("system", "Airline support."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])
bad_chain = bad_prompt | model | StrOutputParser()
store_A6 = {}
def get_hist_A6(sid):
    if sid not in store_A6:
        store_A6[sid] = InMemoryChatMessageHistory()
    return store_A6[sid]

bot_A6 = RunnableWithMessageHistory(
    bad_chain, get_hist_A6,
    input_messages_key="question",        # matches ("human", "{question}")
    history_messages_key="chat_history",  # matches MessagesPlaceholder("chat_history")
)
cfg = {"configurable": {"session_id": "cust-rao"}}
bot_A6.invoke({"question": "I am Rao, PNR JX48Q2."}, config=cfg)
print(bot_A6.invoke({"question": "What is my PNR?"}, config=cfg))

Your PNR is JX48Q2.


In [15]:
# CHECK A6
assert len(store_A6["cust-rao"].messages) == 4, "History is not accumulating; keys still mismatch."
print("A6 passed: keys match, history accumulates.")

A6 passed: keys match, history accumulates.


## Build A7: isolate two customers (write the factory)
**Brief:** stop the cross-customer leak from Derailment 3.

**Decision steps**
1. Keep a dict from `session_id` to a history object.
2. Create a new `InMemoryChatMessageHistory` the first time an id appears.
3. Return the one for that id.

In [15]:
store_A7 = {}

def get_session_history_A7(session_id):
    # TODO: return a per-session InMemoryChatMessageHistory, creating it on first use.
    raise NotImplementedError

bot_A7 = RunnableWithMessageHistory(
    chain, get_session_history_A7,
    input_messages_key="input", history_messages_key="history")

In [16]:
# SOLUTION A7
store_A7 = {}

def get_session_history_A7(session_id):
    if session_id not in store_A7:
        store_A7[session_id] = InMemoryChatMessageHistory()
    return store_A7[session_id]

bot_A7 = RunnableWithMessageHistory(
    chain, get_session_history_A7,
    input_messages_key="input", history_messages_key="history")

bot_A7.invoke({"input": "I am Rao, PNR JX48Q2. Allergic to peanuts."},
              config={"configurable": {"session_id": "rao"}})
bot_A7.invoke({"input": "I am Mehta, PNR ZZ90Q1."},
              config={"configurable": {"session_id": "mehta"}})
print(bot_A7.invoke({"input": "What is my PNR?"},
                    config={"configurable": {"session_id": "mehta"}}))

Your PNR is ZZ90Q1.


In [17]:
# CHECK A7
assert set(store_A7.keys()) == {"rao", "mehta"}, "Expected two separate sessions."
assert not any("peanut" in m.content.lower() for m in store_A7["mehta"].messages), \
    "Leak: Rao's data reached Mehta."
print("A7 passed: two customers, zero leakage.")

A7 passed: two customers, zero leakage.


## Build A8: reproduce and fix Instance Amnesia
**Brief:** two replicas, each with its own dict, forget at random.

**Decision steps**
1. Run turn 1 on replica A, turn 2 on replica B, and watch the forgetting.
2. Fix it by pointing both at ONE shared store, the way Redis or Postgres would.

In [18]:
def make_bot(store_dict):
    def _get(sid):
        if sid not in store_dict:
            store_dict[sid] = InMemoryChatMessageHistory()
        return store_dict[sid]
    return RunnableWithMessageHistory(chain, _get,
        input_messages_key="input", history_messages_key="history")

replica_A, replica_B = {}, {}
botA, botB = make_bot(replica_A), make_bot(replica_B)

botA.invoke({"input": "I am Rao, PNR JX48Q2."}, config={"configurable": {"session_id": "rao"}})
lost = botB.invoke({"input": "What is my PNR?"}, config={"configurable": {"session_id": "rao"}})
print("Split replicas ->", lost)

# TODO: create ONE shared dict and build two bots that both use it, then repeat the two turns.

Split replicas -> I do not have your PNR. Could you share it?


In [18]:
# SOLUTION A8 (self-contained)
def make_bot(store_dict):
    def _get(sid):
        if sid not in store_dict:
            store_dict[sid] = InMemoryChatMessageHistory()
        return store_dict[sid]
    return RunnableWithMessageHistory(chain, _get,
        input_messages_key="input", history_messages_key="history")

shared_store = {}
botA2, botB2 = make_bot(shared_store), make_bot(shared_store)   # both share one dict

botA2.invoke({"input": "I am Rao, PNR JX48Q2."}, config={"configurable": {"session_id": "rao"}})
fixed = botB2.invoke({"input": "What is my PNR?"}, config={"configurable": {"session_id": "rao"}})
print("Shared store ->", fixed)

Shared store -> Your PNR is JX48Q2.


In [19]:
# CHECK A8
assert "JX48Q2" in fixed, "Still forgetting; the store is not shared."
assert len(shared_store["rao"].messages) == 4, "Both turns should live in one store."
print("A8 passed: shared store removes Instance Amnesia.")

A8 passed: shared store removes Instance Amnesia.


## Build A9: measure the Replay Tax (write the counting loop)
**Brief:** show that cumulative history re-sent grows faster than the number of turns.

**Decision steps**
1. Grow a conversation turn by turn.
2. After each turn, record how many messages are on the notepad.
3. Sum those counts. The running total should climb faster than the turn number.

In [21]:
turns_A9 = ["one", "two", "three", "four", "five"]
convo_A9 = [SystemMessage("Airline support.")]
per_turn = []   # messages present on each turn

# TODO: for each turn, append a human message, record len(convo_A9), append an ai message.
# Fill per_turn with the message count seen on each turn.
raise NotImplementedError

NotImplementedError: 

In [22]:
# SOLUTION A9
turns_A9 = ["one", "two", "three", "four", "five"]
convo_A9 = [SystemMessage("Airline support.")]
per_turn = []

for t in turns_A9:
    convo_A9.append(HumanMessage(t))
    per_turn.append(len(convo_A9))   # what this turn re-sends
    convo_A9.append(AIMessage("ack"))

cumulative = sum(per_turn)
print("messages re-sent per turn:", per_turn)
print("cumulative re-sends      :", cumulative)

messages re-sent per turn: [2, 4, 6, 8, 10]
cumulative re-sends      : 30


In [23]:
# CHECK A9
assert per_turn == sorted(per_turn) and per_turn[-1] > per_turn[0], \
    "Per-turn cost should grow."
assert sum(per_turn) > len(turns_A9) * per_turn[0], \
    "Cumulative cost should outgrow a linear baseline."
print("A9 passed: you measured the quadratic Replay Tax.")

A9 passed: you measured the quadratic Replay Tax.


## Build A10: cap the tax with trimming
**Brief:** keep only the last few messages before each call.

**Decision steps**
1. Use `trim_messages` with `strategy="last"` and `token_counter=len`.
2. Keep the system message.
3. Cap to the last 4 messages.

In [24]:
from langchain_core.messages import trim_messages

long_convo = [SystemMessage("Airline support.")]
for i in range(8):
    long_convo.append(HumanMessage(f"q{i}"))
    long_convo.append(AIMessage(f"a{i}"))

# TODO: produce `trimmed_A10` keeping the system message and the last 4 messages.
trimmed_A10 = None
raise NotImplementedError

NotImplementedError: 

In [20]:
# SOLUTION A10 (self-contained)
from langchain_core.messages import trim_messages
long_convo = [SystemMessage("Airline support.")]
for i in range(8):
    long_convo.append(HumanMessage(f"q{i}"))
    long_convo.append(AIMessage(f"a{i}"))

trimmed_A10 = trim_messages(
    long_convo,
    max_tokens=4,
    strategy="last",
    token_counter=len,
    include_system=True,
)
show("Trimmed to last 4", trimmed_A10)

--- Trimmed to last 4: 4 message(s) the model will see ---
  1. system | Airline support.
  2. ai     | a6
  3. human  | q7
  4. ai     | a7


In [21]:
# CHECK A10
assert trimmed_A10 is not None, "Set trimmed_A10."
assert len(trimmed_A10) <= 5, "Should keep system plus about 4 recent messages."
assert trimmed_A10[0].type == "system", "System message should be kept."
print("A10 passed: history is capped, cost is bounded.")

A10 passed: history is capped, cost is bounded.


## Capstone: an isolated, trimmed copilot on LangGraph
**Brief:** combine identity, memory, and a cost cap on the production path.

**Decision steps**
1. Build a one node LangGraph over `MessagesState` with a `MemorySaver`.
2. Trim inside the node so the model never sees more than the last 6 messages.
3. Prove two `thread_id`s stay isolated.

In [27]:
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import trim_messages

# TODO: implement call_model_cap so it trims state["messages"] to the last 6 (keep system)
# before invoking `model`, then build, compile with MemorySaver, and run two threads.
raise NotImplementedError

NotImplementedError: 

In [28]:
# SOLUTION Capstone (self-contained)
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import trim_messages

def call_model_cap(state):
    recent = trim_messages(state["messages"], max_tokens=6, strategy="last",
                           token_counter=len, include_system=True)
    return {"messages": [model.invoke(recent)]}

b = StateGraph(MessagesState)
b.add_node("agent", call_model_cap)
b.add_edge(START, "agent")
app = b.compile(checkpointer=MemorySaver())

rao = {"configurable": {"thread_id": "rao"}}
mehta = {"configurable": {"thread_id": "mehta"}}

app.invoke({"messages": [HumanMessage("I am Rao, PNR JX48Q2. Allergic to peanuts.")]}, config=rao)
app.invoke({"messages": [HumanMessage("I am Mehta, PNR ZZ90Q1.")]}, config=mehta)
ans = app.invoke({"messages": [HumanMessage("What is my PNR?")]}, config=mehta)
print("Mehta ->", ans["messages"][-1].content)

Mehta -> Your PNR is ZZ90Q1.


In [29]:
# CHECK Capstone
rao_state = app.get_state({"configurable": {"thread_id": "rao"}})
mehta_state = app.get_state({"configurable": {"thread_id": "mehta"}})
rao_text = " ".join(m.content for m in rao_state.values["messages"])
mehta_text = " ".join(m.content for m in mehta_state.values["messages"])
assert "peanut" in rao_text and "peanut" not in mehta_text, "Threads are not isolated."
print("Capstone passed: isolated threads, trimmed context, durable-ready path.")

Capstone passed: isolated threads, trimmed context, durable-ready path.


## Annotate this before you leave

The production target you are building toward. For each box, say which activity taught it.

```mermaid
flowchart TB
    GW[API gateway and auth] --> ID[Identity issues thread id]
    ID --> ORCH[LangGraph orchestration]
    ORCH --> LLM[Claude via Bedrock]
    ORCH --> CTX[Trim and summarize]
    ORCH --> CKPT[(Postgres or Redis checkpointer)]
    ORCH --> OBS[Traces and cost]
```

| Box | Activity that taught it |
|---|---|
| Identity issues thread id | A7 isolation |
| LangGraph orchestration | Capstone |
| Trim and summarize | A9 and A10 |
| Postgres or Redis checkpointer | A8 shared store |
| Traces and cost | A9 Replay Tax |

**Last skeptic question to sit with:** if the model has no memory of its own, what did you actually build in this notebook? One sentence, in your own words.